In [0]:
import dataiku
import pandas as pd
import json
import os
import glob
from datetime import datetime

def get_agent_usage_from_disk(days_back=30):
    # 1. Locate the Data Directory using environment variables
    dss_data_dir = os.environ.get("DIP_HOME")
    if not dss_data_dir:
        raise Exception("Could not determine DSS Data Directory (DIP_HOME).")
    
    audit_log_path = os.path.join(dss_data_dir, "run", "audit")
    print(f"Scanning audit logs in: {audit_log_path}")

    # 2. Identify log files (audit.log is current, audit.log-YYYY-MM-DD are rotated)
    # We grab all of them to ensure we cover the requested time window
    #all_log_files = glob.glob(os.path.join(audit_log_path, "audit.log*"))
    all_log_files = glob.glob(audit_log_path)
    print(f"All log files: {all_log_files}")
    
    usage_data = []
    min_timestamp = (datetime.now().timestamp() - (days_back * 86400)) * 1000

    # 3. Iterate and Parse
    for log_file in all_log_files:
        try:
            print(f"Opening log file {log_file}")
            with open(log_file, 'r', encoding='utf-8') as f:
                for line in f:
                    try:
                        event = json.loads(line)
                        
                        # Filter by timestamp first (optimization)
                        if event.get('timestamp', 0) < min_timestamp:
                            continue

                        # 4. Filter for LLM Mesh events
                        # The 'topic' key usually identifies LLM traffic. 
                        # Common topics: 'llm-mesh-query', 'external-model-query'
                        topic = event.get('topic', '')
                        if 'llm' not in topic and 'external-model' not in topic:
                            continue
                        
                        # Extract payload
                        data = event.get('data', {})
                        
                        # Skip failed calls (optional, depends if you pay for failed calls)
                        if data.get('outcome') != 'SUCCESS':
                            continue

                        details = data.get('details', {})
                        usage = data.get('usage', {})
                        context = data.get('context', {})

                        # Heuristic to find Agent Name (fallback to ID)
                        agent_name = context.get('agentName') or details.get('agentName') or "Direct/Unknown"
                        llm_model = details.get('llmId') or data.get('target', {}).get('llmId')

                        usage_data.append({
                            "Timestamp": datetime.fromtimestamp(event.get('timestamp', 0)/1000),
                            "Agent Name": agent_name,
                            "LLM Model": llm_model,
                            "Cost ($)": usage.get('estimatedCost', 0.0),
                            "Total Tokens": usage.get('totalTokens', 0),
                            "User": event.get('auth', {}).get('user')
                        })

                    except json.JSONDecodeError:
                        continue # Skip malformed lines
        except PermissionError:
            print(f"Permission denied reading {log_file}. Ensure code runs as 'dssuser'.")

    # 5. Aggregate
    if not usage_data:
        print("No LLM events found in local audit logs.")
        return pd.DataFrame()

    df = pd.DataFrame(usage_data)
    
    # Summary pivot
    summary = df.groupby(['Agent Name', 'LLM Model']).agg({
        'Cost ($)': 'sum',
        'Total Tokens': 'sum',
        'Timestamp': 'count'
    }).rename(columns={'Timestamp': 'Call Count'}).reset_index()
    
    return summary.sort_values(by='Cost ($)', ascending=False)

# --- Execution ---
try:
    df_usage = get_agent_usage_from_disk(days_back=30)
    print("--- Agent Utilization Report (From Disk) ---")
    print(df_usage.to_string(index=False))
except Exception as e:
    print(f"Failed to generate report: {e}")

In [0]:
import os
import glob

def troubleshoot_audit_path():
    # 1. Verify DIP_HOME
    dip_home = os.environ.get("DIP_HOME")
    print(f"DEBUG: DIP_HOME Env Var = {dip_home}")
    
    if not dip_home:
        print("ERROR: DIP_HOME is not set. Cannot locate data directory.")
        return

    # 2. Check 'run' directory
    run_dir = os.path.join(dip_home, "run")
    if not os.path.exists(run_dir):
        print(f"ERROR: 'run' directory not found at {run_dir}")
        return
    
    # 3. Check 'audit' directory specifically
    audit_dir = os.path.join(run_dir, "audit")
    print(f"DEBUG: Checking Audit Dir = {audit_dir}")
    
    if os.path.exists(audit_dir):
        print("SUCCESS: Audit directory exists.")
        
        # List all files (not just .log) to see if naming convention is different
        all_files = os.listdir(audit_dir)
        print(f"DEBUG: File count in {audit_dir}: {len(all_files)}")
        
        if len(all_files) > 0:
            print(f"DEBUG: First 5 files found: {all_files[:5]}")
        else:
            print("WARNING: Audit directory is empty.")
    else:
        print("ERROR: Audit directory does NOT exist.")
        
        # 4. If 'audit' is missing, list what IS in 'run' to verify structure
        print(f"DEBUG: Contents of {run_dir}:")
        print(os.listdir(run_dir))

troubleshoot_audit_path()

In [0]:
/data/dataiku/dss_data/run